In [14]:
sims_apo = "/Users/aspomme/Desktop/Master_arbeit/FULL-LHGCR/Apo_7FIJ/md_runs"
sims_holo = "/Users/aspomme/Desktop/Master_arbeit/FULL-LHGCR/Holo_7FIH/md_runs"

In [21]:
#emoved in version 2.1, so it is no longer available. Here's an updated version of the previous script that uses the MDAnalysis.lib.distances module instead:

import numpy as np
import MDAnalysis as mda
from MDAnalysis.lib.distances import distance_array
import numpy as np
from MDAnalysis.analysis import distances

In [35]:
import MDAnalysis as mda
from MDAnalysis.analysis import distances
import numpy as np

# Load the universe and select the atoms by their serial number
u = mda.Universe(f'{sims_holo}/md_lite.tpr', f'{sims_holo}/md_1ms.xtc')
atom_pairs = [(3627, 3759), (4779, 5908), (4771,8831),(4767,5972),(3260,6047),(3265,5989),(3759,4767),(4718,8831),(4783,8831),
(2758,4309),(4767,8831),(4711,7282)] # Add all 12 pairs of atom serial numbers here
atoms = [u.select_atoms(f'bynum {serial_number}') for pair in atom_pairs for serial_number in pair]

# Calculate the distance between each pair of atoms for each frame
dist_list = [[] for _ in range(len(atom_pairs))]
for ts in u.trajectory:
    for i, (atom1, atom2) in enumerate(zip(atoms[::2], atoms[1::2])):
        distance = distances.distance_array(atom1.positions, atom2.positions)
        dist_list[i].append(distance[0][0])


# Calculate the mean distance and standard deviation for each pair of atoms
mean_distances = []
std_distances = []
for i, pair in enumerate(atom_pairs):
    mean_distance = np.mean(dist_list[i])
    std_distance = np.std(dist_list[i])
    mean_distances.append(mean_distance)
    std_distances.append(std_distance)
    
#save
# Combine the mean and standard deviation values for each pair of atoms into a single array
pair_values = np.zeros((len(mean_distances), 2))
pair_values[:, 0] = mean_distances
pair_values[:, 1] = std_distances

# Save the pair values to a binary file using NumPy's save function
np.save('Holo_values.npy', pair_values)


In [36]:
Holo_values = np.load('Holo_values.npy')
print(Holo_values)

[[15.71685946  2.47914161]
 [11.41780912  2.76990143]
 [ 6.25610885  0.91707166]
 [10.6375131   1.09954757]
 [ 7.10554365  3.97144587]
 [ 7.48694858  2.40228867]
 [22.41831735  2.89626425]
 [ 9.64295357  2.77003798]
 [ 4.39112406  1.01099053]
 [34.70511002  7.55429921]
 [ 5.70394559  2.33130702]
 [ 7.76248129  1.0788721 ]]


In [42]:
u = mda.Universe(f'{sims_apo}/md_Apo.tpr', f'{sims_apo}/md_1-1000_cm.xtc')
atom_pairs = [(3627, 3759), (4779, 5908), (4771,8831),(4767,5972),(3260,6047),(3265,5989),(3759,4767),(4718,8831),(4783,8831),
(2758,4309),(4767,8831),(4711,7282)] # Add all 12 pairs of atom serial numbers here
atoms = [u.select_atoms(f'bynum {serial_number}') for pair in atom_pairs for serial_number in pair]

# Calculate the distance between each pair of atoms for each frame
dist_list = [[] for _ in range(len(atom_pairs))]
for ts in u.trajectory:
    for i, (atom1, atom2) in enumerate(zip(atoms[::2], atoms[1::2])):
        distance = distances.distance_array(atom1.positions, atom2.positions)
        dist_list[i].append(distance[0][0])

# Calculate the mean distance and standard deviation for each pair of atoms
mean_distances = []
std_distances = []
for i, pair in enumerate(atom_pairs):
    mean_distance = np.mean(dist_list[i])
    std_distance = np.std(dist_list[i])
    mean_distances.append(mean_distance)
    std_distances.append(std_distance)

# Combine the mean and standard deviation values for each pair of atoms into a single array
pair_values = np.zeros((len(mean_distances), 2))
pair_values[:, 0] = mean_distances
pair_values[:, 1] = std_distances

# Save the pair values to a binary file using NumPy's save function
np.save('Apo_values.npy', pair_values)

In [43]:
Apo_values = np.load('Apo_values.npy')
print(Apo_values)

[[16.11110991  2.62597022]
 [ 7.57285821  5.17113961]
 [ 7.1222441   1.39565589]
 [11.04881509  3.06968653]
 [ 5.35896408  3.12967423]
 [ 9.22071649  3.30283022]
 [17.54414121  3.92059183]
 [ 8.40036899  1.23187847]
 [ 7.40003797  1.9442518 ]
 [ 4.35131009  2.27557974]
 [10.58754157  1.90524577]
 [10.4875361   1.3587455 ]]


In [61]:
import numpy as np
import pandas as pd
# Define the data
Apo_values = np.load('Apo_values.npy')
Holo_values = np.load('Holo_values.npy')

# Calculate the HLDA scores
hlda_scores = ((Apo_values[:, 0] - Holo_values[:, 0])) * (1 / Apo_values[:, 1] + 1 / Holo_values[:, 1])

# Create a table with the results
results = pd.DataFrame(hlda_scores.reshape(1,-1), columns=['d'+str(i+1) for i in range(12)])

# Print the table
print(results)

         d1        d2        d3        d4        d5        d6        d7  \
0  0.309162 -2.131659  1.565051  0.508053 -0.997855  1.246649 -2.926143   

         d8        d9        d10       d11       d12  
0 -1.457271  4.523799 -17.357013  4.658026  4.531404  
